# Préparation SFT — MediQAl, QCM à une seule bonne réponse (mcqu)

QCM médicaux en français, avec leur corrigé (CC-BY-4.0). Ils apportent des connaissances, et une décision sur un
choix fermé, facile à évaluer.

- **Une conversation par cas clinique** : le cas, puis les questions dans leur ordre (identifiant croissant), chacune
  avec ses propositions A à E et sa consigne, suivie de la bonne réponse (« C. texte »). Une ligne par cas
  (coupée en tranches s'il est trop long) ; à l'entraînement, toutes les réponses sont apprises
  (`assistant_only_loss`). Une question sans cas donne une ligne d'un seul échange.
- **Sélection pour le triage** : cas avec constantes ou vocabulaire d'urgence, dans un budget de questions
  (`projet_14.selection`), avec un échantillon exporté pour relecture.
- **Format commun** à tous les notebooks v4 : `projet_14.conversation`.
- **Sortie** : `data/v4/sft_mediqal_mcqu_{train,validation,test}.jsonl`, puis la version anonymisée dans `data/v4/anonymized/`.

# Import

In [1]:
import re
from pathlib import Path

import pandas as pd
from datasets import Dataset, load_dataset

from projet_14.conversation import CASE_LABEL, conversation_windows, split_of
from projet_14.mediqal import LETTERS, OPTIONS_HEADER, case_id, clean_text, correct_letters, format_answer
from projet_14.rgpd import run
from projet_14.selection import case_table, review_sample, select_cases

OUT = Path("../data/v4")
CONFIG, MULTIPLE = "mcqu", False
SOURCE, STEM = f"MediQAl-{CONFIG}", f"sft_mediqal_{CONFIG}"
MAX_EXAMPLE_CHARS = 4_000  # ~1 200 tokens : sous le max_length du SFT (1 536)

# Sélection pour le triage (environ 3 000 questions en français : oeq 1 100, mcqu 1 200, mcqm 700)
BUDGET = 1_200            # questions retenues, tous splits confondus
MAX_SUBJECT_SHARE = 0.15  # part maximale du budget par spécialité
REVIEW_SAMPLE = 30        # cas exportés pour relecture

/home/johan/projects/P14/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Chargement

Les trois parties officielles sont réunies : leur découpage partage des cas cliniques entre train et test. On refait
le découpage par cas clinique (plus bas).

In [2]:
ds = load_dataset("ANR-MALADES/MediQAl", CONFIG)
df = pd.concat([ds[name].to_pandas() for name in ds], ignore_index=True)
df.shape

Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


Ignoring part of dataset_info from the dataset card: The `splits` field in YAML is malformed or missing the name/num_examples/num_bytes fields


(17017, 12)

# Analyse

In [3]:
print(df["medical_subject"].value_counts().head(10), "\n")
print(f"avec cas clinique : {df['clinical_case'].notna().mean():.0%}")
df.head()

medical_subject
Cardiology                      1192
Hepato-Gastroenterology         1106
Gynecology and Obstetrics       1063
Hematology                       974
Pulmonology                      917
Endocrinology and Metabolism     898
Nephro-Urology                   848
Pediatrics                       800
Rheumatology                     703
Infectious Diseases              681
Name: count, dtype: int64 

avec cas clinique : 41%


,id,clinical_case,question,answer_a,answer_b,answer_c,answer_d,answer_e,correct_answers,task,medical_subject,question_type
0,7543,"Monsieur R. âgé de 37 ans, forestier de son mé...",Au sujet des vaccinations :,Le vaccin contre la fièvre jaune est obligatoire,Le vaccin contre la fièvre jaune peut se faire...,Le vaccin contre le choléra est un vaccin qui ...,Le vaccin contre l'hépatite B est conseillé ma...,Les vaccinations contre le tétanos et la polio...,A,QCU,Infectious Diseases,Understanding
1,10495,NaN,"Devant une tuméfaction cervicale médiane, juxt...",Kyste dermoïde,Laryngocèle,Kyste du tractus thyréoglosse,Grenouillette sublinguale,Lithiase salivaire,C,QCU,Otorhinolaryngology (ENT),Understanding
2,17342,Une jeune femme de 24 ans vous consulte pour a...,Le traitement de cette patiente fait appel à :,Chirurgie en urgence,Iode radio-actif en urgence,Iode radio-actif en préparation à la chirurgie,Administration d'antithyroïdiens,Aucun de ces traitements,D,QCU,Endocrinology and Metabolism,Reasoning
3,22883,"Un nouveau-né de sexe masculin, âgé de 3 jours...",Chez ce nouveau-né le bilan biologique à montr...,Ictère par Incompatibilité Rhésus,Ictère par déficit en G6PD,ictère infectieux,Ictère par incompatibilité ABO,Ictère Infectieux,D,QCU,Pediatrics,Understanding
4,17514,Un homme de 63 ans est hospitalisé pour des do...,Pour confirmer le diagnostic de maladie d'Addi...,Test de stimulation par la 1.24 corticotrophin...,Test de freinage par la dexaméthasone,Test de stimulation de l'activité rénine plasm...,Test de restriction hydrique,Scintigraphie surrénale à l'iodocholestérol,A,QCU,Endocrinology and Metabolism,Reasoning


# Nettoyage

- Espaces et lignes vides en trop ; doublons retirés.
- La mention « (cochez la réponse juste) » est retirée : la consigne en fin de message la remplace.
  **« (cochez la réponse fausse) » est gardée** : elle change le sens de la question.

In [4]:
TEXT_COLUMNS = ["clinical_case", "question"] + [f"answer_{letter.lower()}" for letter in LETTERS]
for col in TEXT_COLUMNS:
    df[col] = df[col].map(clean_text, na_action="ignore")
df["clinical_case"] = df["clinical_case"].replace("", None)

JUSTE = re.compile(r"^\(\s*cochez la réponse juste\s*\)\s*", re.I)
print(f"mentions « réponse juste » retirées : {df['question'].str.contains(JUSTE).sum()}")
df["question"] = df["question"].str.replace(JUSTE, "", regex=True)

df = df.dropna(subset=["question", "correct_answers"]).drop_duplicates(subset=["clinical_case", "question"])
df.shape

mentions « réponse juste » retirées : 2627


(16508, 12)

# Filtres

Questions qui renvoient à une image ou un document (« ci-dessous », « figure »…) : la réponse est impossible à
partir du seul texte.

In [5]:
IMAGE = re.compile(r"ci-dessous|ci-contre|ci-joint|figure|l'image|cette image|photo|schéma suivant|document suivant|annexe", re.I)
image = df["question"].str.contains(IMAGE) | df["clinical_case"].fillna("").str.contains(IMAGE)
print(f"renvoient à une image ou un document : {image.sum()}")
df = df[~image]
df.shape

renvoient à une image ou un document : 261


(16247, 12)

# Répartition par cas clinique

Toutes les questions d'un cas vont dans le même split (70 % train, 10 % validation, 20 % test), tiré de
l'identifiant du cas : le même cas tombe dans le même split dans tous les notebooks. Une question sans cas est
répartie seule.

In [6]:
key = df["clinical_case"].fillna(df["question"])  # le cas, ou la question elle-même s'il n'y a pas de cas
df["case_id"] = key.map(case_id)
df["split"] = key.map(split_of)

# Doublons à la casse ou aux espaces près (même question dans le même cas) : un seul gardé
question_norm = df["question"].str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
duplicate = pd.concat([df["case_id"], question_norm], axis=1).duplicated()
print(f"doublons à la casse ou aux espaces près : {duplicate.sum()}")
df = df[~duplicate]
pd.crosstab(df["split"], df["clinical_case"].notna().map({True: "avec cas", False: "sans cas"}), margins=True)

doublons à la casse ou aux espaces près : 35


clinical_case,avec cas,sans cas,All
split,,,
test,1433,1852,3285
train,4614,6666,11280
validation,718,929,1647
All,6765,9447,16212


# Conversations par cas

Les questions d'un cas sont triées par identifiant (l'ordre du récit), et le cas devient **une ligne** : une
conversation complète, coupée en tranches s'il est trop long. Réponse attendue : « C. texte ».

In [7]:
df["order"] = pd.to_numeric(df["id"], errors="coerce")

rows = []
for cid, group in df.sort_values("order").groupby("case_id", sort=False):
    case = group["clinical_case"].iloc[0]
    case = case if isinstance(case, str) else None  # question sans cas (valeur manquante : NaN)
    turns = []
    for r in group.itertuples():
        row = r._asdict()
        options = {letter: row[f"answer_{letter.lower()}"] for letter in LETTERS if row[f"answer_{letter.lower()}"]}
        turns.append({"question": r.question, "options": options, "multiple": MULTIPLE,
                      "answer": format_answer(row, correct_letters(row), as_list=MULTIPLE)})
    for start, messages in conversation_windows(case, turns):
        part = group.iloc[start:start + (len(messages) - 1) // 2]  # questions de cette ligne
        rows.append({
            "ids": [str(i) for i in part["id"]], "source": SOURCE, "case_id": cid, "split": part["split"].iloc[0],
            "n_questions": len(part), "medical_subject": part["medical_subject"].iloc[0],
            "question_types": list(part["question_type"]), "golden_answers": list(part["correct_answers"]),
            "lang": "fr", "messages": messages,
        })
examples = pd.DataFrame(rows)

# Une question seule déjà trop longue (cas très long) : écartée plutôt que coupée à l'entraînement
examples["chars"] = examples["messages"].map(lambda m: sum(len(x["content"]) for x in m))
too_long = examples["chars"] > MAX_EXAMPLE_CHARS
print(f"{len(examples)} lignes ({examples['n_questions'].sum()} questions), dont {too_long.sum()} trop longues écartées")
examples = examples[~too_long]
examples.groupby("split").agg(lignes=("n_questions", "size"), questions=("n_questions", "sum"))

11553 lignes (16212 questions), dont 0 trop longues écartées


,lignes,questions
split,,
test,2305,3285
train,8107,11280
validation,1141,1647


In [8]:
# Une ligne à plusieurs questions, telle que la verra le modèle
example = examples[examples["n_questions"] >= 2].iloc[0]
for message in example["messages"]:
    print(f"--- {message['role']}\n{message['content'][:500]}\n")

--- system
Tu es un assistant médical expert. Réponds de façon précise et concise.

--- user
Cas clinique : Une femme de 40 ans, non ménopausée, présente une tuméfaction du sein gauche, connue depuis plusieurs mois. A l'examen clinique, cette tuméfaction fait 4 cm de diamètre, elle n'est adhérente ni à la peau, ni aux plans profonds. La palpation des aires ganglionnaires retrouve deux ganglions axillaires gauches mobiles, non inflammatoires, indolores, suspects. Enfin, le bilan paraclinique d'extension s'avère négatif. Une drill-biopsie confirme qu'il s'agit d'un épithélioma glandulaire

--- assistant
A. T2 N1b M0

--- user
Question : Compte-tenu des résultats histologiques et biochimiques de la pièce opératoire, quel schéma thérapeutique logique vous paraît le mieux adapté parmi ceux proposé après la mammectomie :

Propositions :
A. Surveillance régulière seule
B. Irradiation loco-régionale seule
C. Hormonothérapie adjuvante seule
D. Irradiation loco-régionale + hormonothérapie adjuva

# Sélection pour le triage

Le corpus complet (~30 000 questions en français) dépasse le besoin : environ 5 000 paires SFT, dont ~3 000 en
français. On garde les **cas les plus proches du triage**, par cas entier (`projet_14.selection`) :

1. **Cas clinique obligatoire** : le triage part toujours d'un patient décrit.
2. **Score de pertinence** (0 à 2) : **constantes** dans le cas (PA, FC, SpO2, température, Glasgow…) +
   vocabulaire d'**urgence** ou de conduite à tenir (urgence, hospitalisation, gravité, choc, détresse…).
3. **Budget** de `BUDGET` questions, réparti entre les splits (70 / 10 / 20) : dans chaque split, cas pris par score
   décroissant (score ≥ 1), puis dans un ordre pseudo-aléatoire fixe tiré de l'identifiant du cas.
4. **Plafond par spécialité** : `MAX_SUBJECT_SHARE` du budget au plus, pour garder la diversité.

Le même tri s'applique aux trois splits : le test ressemble au train.

In [9]:
cases = case_table(examples, lang="fr", group="medical_subject")
# Questions disponibles selon la présence d'un cas et le score
print(pd.crosstab(cases["patient"].map({True: "avec cas", False: "sans cas"}), cases["score"],
                  values=cases["n_questions"], aggfunc="sum", margins=True).fillna(0).astype(int), "\n")

selected = select_cases(cases, BUDGET, MAX_SUBJECT_SHARE, group="medical_subject")
examples = examples[examples["case_id"].isin(selected["case_id"])]
print(f"retenus : {len(selected)} cas, {selected['n_questions'].sum()} questions (budget {BUDGET})")
selected.groupby("split").agg(cas=("case_id", "size"), questions=("n_questions", "sum"),
                              constantes=("constantes", "mean"), urgence=("urgence", "mean")).round(2)

score         0     1     2    All
patient                           
avec cas   2335  2838  1592   6765
sans cas   8889   558     0   9447
All       11224  3396  1592  16212 

retenus : 293 cas, 1200 questions (budget 1200)


,cas,questions,constantes,urgence
split,,,,
test,50,240,1.00,1.0
train,211,840,1.00,1.0
validation,32,120,0.94,1.0


In [10]:
# Spécialités retenues (questions) : aucune ne dépasse MAX_SUBJECT_SHARE du budget
selected.groupby("medical_subject")["n_questions"].sum().sort_values(ascending=False)

medical_subject
Hepato-Gastroenterology             158
Cardiology                          155
Pulmonology                         148
Endocrinology and Metabolism        144
Pediatrics                          116
Infectious Diseases                  88
Gynecology and Obstetrics            86
Hematology                           71
Nephro-Urology                       66
Neurology                            36
Orthopedics                          27
Dermatology                          21
Otorhinolaryngology (ENT)            20
Psychiatry                           19
Forensic Medicine and Toxicology     15
Rheumatology                         12
Immunology                            9
Ophthalmology                         4
Biochemistry                          2
Parasitology                          2
Pharmacology                          1
Name: n_questions, dtype: int64

## Échantillon à relire

`REVIEW_SAMPLE` cas retenus, tirés au hasard, exportés pour une relecture clinique (colonne `pertinent_triage` à
remplir : oui / non). Le fichier contient le texte avant anonymisation : il reste dans `data/`, non versionné.

In [11]:
REVIEW_DIR = OUT / "review"
REVIEW_DIR.mkdir(parents=True, exist_ok=True)
review = review_sample(examples, REVIEW_SAMPLE)
review.to_csv(REVIEW_DIR / f"{STEM}_review.csv", index=False)
review.head(3)

,case_id,split,extrait,reponse,pertinent_triage,commentaire
4236,bd79745e902d,train,"Cas clinique : R... Xavier, né à terme, avec u...",B. Hyperthermie.,,
8901,098404ca35ba,validation,"Cas clinique : Une patiente âgée de 59 ans, co...",C. Palpation des aires ganglionnaires,,
2282,ad165aa92828,train,"Cas clinique : Une femme de 70 ans, vivant seu...",E. Fracture du col du fémur,,


# Export JSONL

In [12]:
OUT.mkdir(parents=True, exist_ok=True)
METADATA = ["ids", "source", "case_id", "n_questions", "medical_subject", "question_types", "golden_answers", "lang"]
for split in ("train", "validation", "test"):
    part = examples[examples["split"] == split]
    Dataset.from_pandas(part[METADATA + ["messages"]], preserve_index=False).to_json(
        OUT / f"sft_mediqal_{CONFIG}_{split}.jsonl", force_ascii=False)
    print(split, len(part), "lignes")

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 43.72ba/s]

train 253 lignes


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 262.01ba/s]

validation 37 lignes


Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 149.17ba/s]

test 63 lignes


# Anonymisation des données

Presidio sur les messages **user** uniquement, et seulement avant l'en-tête « Propositions : » (`keep_after`) : le
cas et la question décrivent le patient ; les propositions et les réponses sont des énoncés médicaux rédigés par les
enseignants, où Presidio prendrait des termes médicaux pour des noms.

In [13]:
reports = {
    split: run(OUT / f"sft_mediqal_{CONFIG}_{split}.jsonl", lang="fr", output_dir=OUT / "anonymized",
               roles=["user"], keep_after=OPTIONS_HEADER)
    for split in ("train", "validation", "test")
}
pd.DataFrame(reports).T[["records", "records_modified", "persons_replaced"]]

,records,records_modified,persons_replaced
train,253,86,163
validation,37,15,22
test,63,22,40


# Contrôle final

In [14]:
final = load_dataset("json", data_files={s: str(OUT / "anonymized" / f"sft_mediqal_{CONFIG}_{s}.anonymized.jsonl") for s in ("train", "validation", "test")})
print(final)

# Aucun cas clinique partagé entre train, validation et test
cases = {s: set(d["case_id"]) for s, d in final.items()}
assert not cases["train"] & (cases["validation"] | cases["test"]), "cas partagé entre train et évaluation"
assert not cases["validation"] & cases["test"], "cas partagé entre validation et test"

# Propositions et réponses intactes après anonymisation
for split in final.values():
    for messages in split["messages"]:
        for m in messages[1:]:
            text = m["content"].split(OPTIONS_HEADER, 1)[-1] if m["role"] == "user" else m["content"]
            assert "<PERSON>" not in text, "proposition ou réponse modifiée"
assert all(m[1]["content"].startswith(CASE_LABEL) for d in final.values() for m in d["messages"]), "ligne sans cas clinique"
print("OK : aucun cas partagé entre les splits, propositions et réponses intactes, toutes les lignes partent d'un cas")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 253 examples [00:00, 19583.27 examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating validation split: 37 examples [00:00, 9488.22 examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

Generating test split: 63 examples [00:00, 9444.60 examples/s]

DatasetDict({
    train: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'medical_subject', 'question_types', 'golden_answers', 'lang', 'messages', 'persons_replaced'],
        num_rows: 253
    })
    validation: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'medical_subject', 'question_types', 'golden_answers', 'lang', 'messages', 'persons_replaced'],
        num_rows: 37
    })
    test: Dataset({
        features: ['ids', 'source', 'case_id', 'n_questions', 'medical_subject', 'question_types', 'golden_answers', 'lang', 'messages', 'persons_replaced'],
        num_rows: 63
    })
})
OK : aucun cas partagé entre les splits, propositions et réponses intactes, toutes les lignes partent d'un cas
